# Bond Forward Pricing — From Spot to Forward Yield

The bond forward price is the foundation of T-Lock pricing. This notebook walks through **every step** from spot bond price to forward yield, showing **three methods** to get the forward and how they connect.

## Contents
1. Spot bond pricing (dirty, clean, accrued, YTM)
2. Forward price — Method A: cash-and-carry (repo)
3. Forward price — Method B: discount curve ratio
4. Forward price — Method C: BondForward class
5. Dirty vs clean forward
6. Forward yield (IRR of the forward price)
7. Forward yield curve (across maturities)
8. Carry and roll-down
9. Sensitivity: how repo changes the forward

In [ ]:
import sys, os
_root = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) in ('examples', 'notebooks') else os.getcwd()
sys.path.insert(0, os.path.join(_root, 'python'))

import math
import numpy as np
import matplotlib.pyplot as plt
from datetime import date
from dateutil.relativedelta import relativedelta

from pricebook.bond import FixedRateBond
from pricebook.bond_forward import BondForward, forward_price_repo, blended_repo_rate
from pricebook.discount_curve import DiscountCurve
from pricebook.day_count import DayCountConvention, year_fraction
from pricebook.schedule import Frequency

print('pricebook loaded OK')

## Parameters — change these

In [ ]:
# ============================================================
# BOND
# ============================================================
ISSUE      = date(2024, 2, 15)
MATURITY   = date(2034, 2, 15)   # 10Y
COUPON     = 0.04125             # 4-1/8%
FACE       = 100.0

# ============================================================
# MARKET
# ============================================================
VAL_DATE   = date(2024, 7, 15)
OIS_RATE   = 0.04               # flat OIS curve
REPO_RATE  = 0.035              # bond repo rate

# ============================================================
# FORWARD
# ============================================================
FWD_DATE   = date(2025, 1, 15)   # 6M forward

# ============================================================
# BUILD
# ============================================================
bond = FixedRateBond.treasury_note(ISSUE, MATURITY, COUPON, FACE)
curve = DiscountCurve.flat(VAL_DATE, OIS_RATE)
tau = year_fraction(VAL_DATE, FWD_DATE, DayCountConvention.ACT_365_FIXED)

print(f'Bond:     {COUPON*100:.3f}% {MATURITY}')
print(f'Val date: {VAL_DATE}')
print(f'Forward:  {FWD_DATE} ({tau:.4f}Y = {(FWD_DATE - VAL_DATE).days}d)')
print(f'OIS:      {OIS_RATE*100:.2f}%')
print(f'Repo:     {REPO_RATE*100:.2f}%')

## 1. Spot Bond Pricing

Starting point: the bond's dirty price, clean price, accrued interest, and yield.

In [ ]:
dirty = bond.dirty_price(curve)
accrued = bond.accrued_interest(VAL_DATE)
clean = dirty - accrued
ytm = bond.yield_to_maturity(dirty, VAL_DATE)

print(f'Dirty price:      {dirty:>10.4f}')
print(f'Accrued interest: {accrued:>10.4f}')
print(f'Clean price:      {clean:>10.4f}')
print(f'YTM:              {ytm*100:>10.4f}%')
print()
print(f'Dirty = Clean + Accrued')
print(f'{dirty:.4f} = {clean:.4f} + {accrued:.4f}  ✓ = {clean + accrued:.4f}')

## 2. Forward Price — Method A: Cash-and-Carry

The no-arbitrage forward price from repo financing:

$$F_{\text{dirty}} = P_{\text{dirty}} \times (1 + r_{\text{repo}} \times \tau) - \text{coupons\_received}$$

Buy the bond today, finance at repo rate, collect coupons along the way.

In [ ]:
# Financing cost
financing_cost = dirty * REPO_RATE * tau

# Coupons received between val and forward date
coupons_received = 0.0
for cf in bond.coupon_leg.cashflows:
    if VAL_DATE < cf.payment_date <= FWD_DATE:
        coupons_received += cf.amount / FACE * 100.0
        print(f'  Coupon: {cf.payment_date}  ${cf.amount / FACE * 100:.4f}')

# Forward dirty price
fwd_dirty_A = dirty * (1 + REPO_RATE * tau) - coupons_received

# Accrued at forward date
accrued_fwd = bond.accrued_interest(FWD_DATE)
fwd_clean_A = fwd_dirty_A - accrued_fwd

print(f'')
print(f'Method A: Cash-and-Carry')
print(f'  Spot dirty:         {dirty:>10.4f}')
print(f'  + Financing cost:   {financing_cost:>10.4f}  ({REPO_RATE*100:.2f}% × {tau:.3f}Y)')
print(f'  - Coupons received: {coupons_received:>10.4f}')
print(f'  ─────────────────────────────')
print(f'  Forward dirty:      {fwd_dirty_A:>10.4f}')
print(f'  - Accrued at fwd:   {accrued_fwd:>10.4f}')
print(f'  = Forward clean:    {fwd_clean_A:>10.4f}')

## 3. Forward Price — Method B: Discount Curve Ratio

Under risk-neutral pricing, the forward price is the spot price grown at the forward rate:

$$F_{\text{dirty}} = P_{\text{dirty}} \times \frac{D(0, t_{\text{val}})}{D(0, t_{\text{fwd}})}$$

This uses the OIS curve, not repo. The difference = the repo-OIS basis.

In [ ]:
# Forward via discount factors
df_val = 1.0  # DF at valuation = 1 (it's the reference date)
df_fwd = curve.df(FWD_DATE)
fwd_rate_ois = -math.log(df_fwd) / tau  # continuously compounded

# This gives the OIS-implied forward (no repo basis)
fwd_dirty_B = dirty * (1 + fwd_rate_ois * tau) - coupons_received
fwd_clean_B = fwd_dirty_B - accrued_fwd

print(f'Method B: Discount Curve Ratio')
print(f'  DF to forward:      {df_fwd:>10.6f}')
print(f'  Implied OIS rate:   {fwd_rate_ois*100:>10.4f}%')
print(f'  Forward dirty:      {fwd_dirty_B:>10.4f}')
print(f'  Forward clean:      {fwd_clean_B:>10.4f}')
print()
print(f'Comparison:')
print(f'  Method A (repo):    {fwd_dirty_A:>10.4f}  (repo = {REPO_RATE*100:.2f}%)')
print(f'  Method B (OIS):     {fwd_dirty_B:>10.4f}  (OIS  = {fwd_rate_ois*100:.2f}%)')
print(f'  Difference:         {fwd_dirty_A - fwd_dirty_B:>10.4f}  (repo-OIS basis effect)')

## 4. Forward Price — Method C: BondForward Class

The library's `BondForward` does everything in one call.

In [ ]:
bf = BondForward(bond, settlement=VAL_DATE, delivery=FWD_DATE, repo_rate=REPO_RATE)
result = bf.price(curve)

print(f'Method C: BondForward class')
print(f'  Forward dirty:      {result.forward_dirty:>10.4f}')
print(f'  Forward clean:      {result.forward_clean:>10.4f}')
print(f'  Spot dirty:         {result.spot_dirty:>10.4f}')
print(f'  Carry:              {result.carry:>10.4f}  (coupon - repo cost)')
print(f'  Repo cost:          {result.repo_cost:>10.4f}')
print(f'  Coupon income:      {result.coupon_income:>10.4f}')
print(f'  Forward DV01:       {result.forward_dv01:>10.4f}')
print()
print(f'Cross-check vs Method A:')
print(f'  BondForward dirty:  {result.forward_dirty:.4f}')
print(f'  Method A dirty:     {fwd_dirty_A:.4f}')
print(f'  Match:              {abs(result.forward_dirty - fwd_dirty_A) < 0.01}')

## 5. Dirty vs Clean Forward — Why It Matters

The dirty forward jumps at coupon dates (accrued resets). The clean forward is smooth.

In [ ]:
fwd_dates = [VAL_DATE + relativedelta(days=d) for d in range(1, 365)]
fwd_dirty_series = []
fwd_clean_series = []
days_series = []

for fd in fwd_dates:
    if fd >= MATURITY:
        break
    t = year_fraction(VAL_DATE, fd, DayCountConvention.ACT_365_FIXED)
    if t < 0.001:
        continue
    try:
        bf_i = BondForward(bond, VAL_DATE, fd, REPO_RATE)
        r = bf_i.price(curve)
        fwd_dirty_series.append(r.forward_dirty)
        fwd_clean_series.append(r.forward_clean)
        days_series.append((fd - VAL_DATE).days)
    except:
        pass

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(days_series, fwd_dirty_series, lw=1.5, color='steelblue', label='Forward dirty')
ax1.plot(days_series, fwd_clean_series, lw=1.5, color='darkorange', label='Forward clean')
ax1.axhline(dirty, ls=':', color='gray', alpha=0.5, label=f'Spot dirty = {dirty:.2f}')
ax1.set_xlabel('Days to delivery')
ax1.set_ylabel('Price')
ax1.set_title('Forward Price Term Structure')
ax1.legend(fontsize=9)
ax1.grid(alpha=0.3)

# Carry profile
carry_series = [d - c for d, c in zip(fwd_dirty_series, fwd_clean_series)]
ax2.plot(days_series, carry_series, lw=1.5, color='seagreen')
ax2.set_xlabel('Days to delivery')
ax2.set_ylabel('Accrued at delivery')
ax2.set_title('Accrued Interest at Forward Date')
ax2.grid(alpha=0.3)

fig.tight_layout()
plt.show()

## 6. Forward Yield

The forward yield is the IRR of the forward bond price — the yield that, applied to the bond's remaining cashflows from the forward date, reproduces the forward price.

$$\text{Forward IRR: solve } P_{\text{fwd}} = \sum_i \frac{c_i}{(1 + y/2)^{n_i}} + \frac{100}{(1 + y/2)^{n_{\text{mat}}}} \text{ for } y$$

In [ ]:
from pricebook.bond_yield import bond_irr, bond_price_from_yield, bond_risk_factor

# Get the accrual schedule from the forward date onward
alphas, times, T_mat = bond.accrual_schedule(FWD_DATE)

# Forward price per unit face
fwd_per_unit = result.forward_dirty / 100.0

# Solve for the yield that reprices the forward
fwd_irr = bond_irr(fwd_per_unit, COUPON, alphas)

# Verify: price from this yield should match forward
check_price = bond_price_from_yield(COUPON, alphas, fwd_irr)

print(f'Forward Yield (IRR)')
print(f'  Forward dirty price:  {result.forward_dirty:.4f}')
print(f'  Per unit face:        {fwd_per_unit:.6f}')
print(f'  Forward IRR:          {fwd_irr*100:.4f}%')
print(f'  Spot YTM:             {ytm*100:.4f}%')
print(f'  Yield change:         {(fwd_irr - ytm)*10000:+.2f}bp')
print()
print(f'Verification:')
print(f'  Price from fwd IRR:   {check_price:.6f}')
print(f'  Actual fwd price:     {fwd_per_unit:.6f}')
print(f'  Match:                {abs(check_price - fwd_per_unit) < 1e-6}')
print()
print(f'Risk factor at forward yield:')
rf = bond_risk_factor(COUPON, alphas, fwd_irr)
print(f'  RF = -dP/dy = {rf:.4f}')
print(f'  Meaning: 1bp yield move ≈ {rf * 0.0001 * 100:.4f} price points')

## 7. Forward Yield Curve

How the forward yield evolves as the delivery date moves further out.

In [ ]:
horizons_months = [1, 2, 3, 6, 9, 12, 18, 24]
fwd_yields = []
fwd_carries = []
labels = []

print(f'{"Horizon":>10s} {"Fwd Dirty":>10s} {"Fwd Yield":>10s} {"vs Spot":>10s} {"Carry":>10s}')
print('─' * 55)
for m in horizons_months:
    fd = VAL_DATE + relativedelta(months=m)
    if fd >= MATURITY:
        break
    try:
        bf_i = BondForward(bond, VAL_DATE, fd, REPO_RATE)
        r_i = bf_i.price(curve)
        a_i, _, _ = bond.accrual_schedule(fd)
        y_i = bond_irr(r_i.forward_dirty / 100.0, COUPON, a_i)
        fwd_yields.append(y_i * 100)
        fwd_carries.append(r_i.carry)
        labels.append(f'{m}M')
        diff_bp = (y_i - ytm) * 10000
        print(f'{m:>8d}M {r_i.forward_dirty:>10.4f} {y_i*100:>9.4f}% {diff_bp:>+9.2f}bp {r_i.carry:>10.4f}')
    except:
        pass

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(labels, fwd_yields, 'o-', lw=2, ms=6, color='steelblue')
ax1.axhline(ytm * 100, ls='--', color='red', alpha=0.7, label=f'Spot YTM = {ytm*100:.3f}%')
ax1.set_xlabel('Forward horizon')
ax1.set_ylabel('Forward yield (%)')
ax1.set_title('Forward Yield Term Structure')
ax1.legend(fontsize=9)
ax1.grid(alpha=0.3)

colors = ['seagreen' if c > 0 else 'crimson' for c in fwd_carries]
ax2.bar(labels, fwd_carries, color=colors, alpha=0.8)
ax2.axhline(0, color='k', lw=0.5)
ax2.set_xlabel('Forward horizon')
ax2.set_ylabel('Carry (price points)')
ax2.set_title('Carry to Delivery (coupon − repo cost)')
ax2.grid(alpha=0.3)

fig.tight_layout()
plt.show()

## 8. Carry and Roll-Down

**Carry** = coupon income − financing cost. Positive carry means the bond earns more than it costs to hold.

**Roll-down** = if the yield curve is upward-sloping, a bond "rolls down" to lower yields as it ages, earning capital gains.

In [ ]:
bf_6m = BondForward(bond, VAL_DATE, FWD_DATE, REPO_RATE)
r_6m = bf_6m.price(curve)

annual_coupon = COUPON * FACE
annual_repo_cost = dirty * REPO_RATE
daily_carry = (annual_coupon - annual_repo_cost) / 365

print(f'Carry Analysis ({(FWD_DATE - VAL_DATE).days}d horizon)')
print(f'  Annual coupon:       ${annual_coupon:>8.4f}  ({COUPON*100:.3f}% × {FACE})')
print(f'  Annual repo cost:    ${annual_repo_cost:>8.4f}  ({REPO_RATE*100:.2f}% × {dirty:.2f})')
print(f'  Daily carry:         ${daily_carry:>8.4f}')
print(f'  Period carry:        ${r_6m.carry:>8.4f}')
print(f'  Carry sign:          {"POSITIVE ✓" if r_6m.carry > 0 else "NEGATIVE ✗"}')
print()
print(f'  When coupon > repo cost → positive carry → forward dirty < spot dirty')
print(f'  When coupon < repo cost → negative carry → forward dirty > spot dirty')
print()

# What repo rate makes carry = 0?
breakeven_repo = annual_coupon / dirty
print(f'Breakeven Analysis')
print(f'  Breakeven repo rate: {breakeven_repo*100:.4f}%')
print(f'  Current repo:        {REPO_RATE*100:.4f}%')
print(f'  Carry advantage:     {(breakeven_repo - REPO_RATE)*10000:+.1f}bp')

## 9. Sensitivity: How Repo Changes the Forward

The repo rate is the **drift** of the forward price. Higher repo → higher forward → lower forward yield.

In [ ]:
repo_range = np.linspace(0.0, 0.06, 60)
fwd_dirty_by_repo = []
fwd_yield_by_repo = []
carry_by_repo = []

for rr in repo_range:
    bf_r = BondForward(bond, VAL_DATE, FWD_DATE, rr)
    r_r = bf_r.price(curve)
    fwd_dirty_by_repo.append(r_r.forward_dirty)
    carry_by_repo.append(r_r.carry)
    try:
        a_r, _, _ = bond.accrual_schedule(FWD_DATE)
        y_r = bond_irr(r_r.forward_dirty / 100.0, COUPON, a_r)
        fwd_yield_by_repo.append(y_r * 100)
    except:
        fwd_yield_by_repo.append(float('nan'))

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].plot(repo_range * 100, fwd_dirty_by_repo, lw=2, color='steelblue')
axes[0].axvline(REPO_RATE * 100, ls='--', color='red', alpha=0.7, label=f'Current = {REPO_RATE*100:.1f}%')
axes[0].set_xlabel('Repo rate (%)')
axes[0].set_ylabel('Forward dirty price')
axes[0].set_title('Forward Price vs Repo')
axes[0].legend(fontsize=9)
axes[0].grid(alpha=0.3)

axes[1].plot(repo_range * 100, fwd_yield_by_repo, lw=2, color='darkorange')
axes[1].axhline(ytm * 100, ls=':', color='gray', alpha=0.5, label=f'Spot YTM = {ytm*100:.3f}%')
axes[1].axvline(REPO_RATE * 100, ls='--', color='red', alpha=0.7)
axes[1].set_xlabel('Repo rate (%)')
axes[1].set_ylabel('Forward yield (%)')
axes[1].set_title('Forward Yield vs Repo')
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)

axes[2].plot(repo_range * 100, carry_by_repo, lw=2, color='seagreen')
axes[2].axhline(0, color='k', lw=0.5)
axes[2].axvline(breakeven_repo * 100, ls=':', color='gray', label=f'Breakeven = {breakeven_repo*100:.2f}%')
axes[2].axvline(REPO_RATE * 100, ls='--', color='red', alpha=0.7, label=f'Current = {REPO_RATE*100:.1f}%')
axes[2].set_xlabel('Repo rate (%)')
axes[2].set_ylabel('Carry (price points)')
axes[2].set_title('Carry vs Repo')
axes[2].legend(fontsize=9)
axes[2].grid(alpha=0.3)

fig.tight_layout()
plt.show()

print(f'Key insight: repo is the DRIFT of the forward.')
print(f'  Higher repo → bond costs more to carry → forward price rises → forward yield falls.')
print(f'  This is why specialness matters: on-special bonds have LOW repo → HIGH forward yield.')